In [1]:
!pip install -q ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 56.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 3.8 MB/s eta 0:00:00


In [2]:
import cv2
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image as PILImage
from ultralytics import YOLO
model = YOLO('yolov8n.pt')

Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [3]:
cap = cv2.VideoCapture("/content/football_jersey_clip.mp4")
ret,frame = cap.read()
ret2, frame2 = cap.read()

In [4]:
def get_centroids(frame):
  result = model.predict(frame)
  boxes = result[0].boxes.xyxy.cpu().numpy()
  conf = result[0].boxes.conf.cpu().numpy()
  centroid=[]

  for box,c in zip(boxes,conf):
    if c < 0.5:
      continue
    x1,y1,x2,y2 = box
    cx = (x1+x2)/2
    cy = (y1+y2)/2
    centroid.append((cx,cy))

  return centroid

In [5]:
centroid_frame1 = get_centroids(frame)
centroid_frame2 = get_centroids(frame2)


0: 384x640 11 persons, 461.8ms
Speed: 26.7ms preprocess, 461.8ms inference, 44.5ms postprocess per image at shape (1, 3, 384, 640)


In [12]:
players_position = {}
next_id = 0
for c in centroid_frame1:
  players_position[next_id] = [c]
  next_id +=1
print(players_position)

{0: [(np.float32(403.42545), np.float32(304.4803))], 1: [(np.float32(99.70708), np.float32(275.8715))], 2: [(np.float32(589.9822), np.float32(353.16367))], 3: [(np.float32(711.9473), np.float32(237.6838))], 4: [(np.float32(146.1174), np.float32(227.2714))], 5: [(np.float32(555.9518), np.float32(242.2549))], 6: [(np.float32(593.5323), np.float32(291.34247))], 7: [(np.float32(403.35294), np.float32(258.49933))]}


In [13]:
import math
cap = cv2.VideoCapture("/content/football_jersey_clip.mp4")   # reopen from frame 0 — the old cap object is exhausted
frame_count = 0
max_frames = 50   # quick test limit — remove once logic is confirmed correct
max_distance = 50

while True:
    ret, frame = cap.read()
    if not ret:
        break
    frame_count += 1
    if frame_count > max_frames:
        break

    centroid = get_centroids(frame)
    for c in centroid:
        best_id = None
        best_distance = float("inf")
        for pid, pos in players_position.items():
          d = math.dist(c, pos[-1])
          if d < best_distance:
              best_id = pid
              best_distance = d
        if best_distance < max_distance:
            players_position[best_id].append(c)   # adds to the list
        else:
            players_position[next_id] = [c]
            next_id += 1

print(players_position)
print(next_id)


0: 384x640 11 persons, 153.3ms
Speed: 6.2ms preprocess, 153.3ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 12 persons, 154.4ms
Speed: 7.3ms preprocess, 154.4ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 12 persons, 172.7ms
Speed: 5.5ms preprocess, 172.7ms inference, 1.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 12 persons, 155.8ms
Speed: 5.4ms preprocess, 155.8ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 12 persons, 157.1ms
Speed: 7.8ms preprocess, 157.1ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 12 persons, 171.1ms
Speed: 4.9ms preprocess, 171.1ms inference, 1.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 12 persons, 157.8ms
Speed: 5.9ms preprocess, 157.8ms inference, 1.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 12 persons, 159.5ms
Speed: 5.4ms preprocess, 159.5ms inference, 1.7ms postproc

In [14]:
player_distances = {}

for pid, history in players_position.items():
    total = 0
    for i in range(len(history) - 1):
        x1, y1 = history[i]
        x2, y2 = history[i+1]
        d = ((x2-x1)**2 + (y2-y1)**2)**0.5
        total += d
    player_distances[pid] = total

print(player_distances)

{0: np.float32(50.36294), 1: np.float32(195.61069), 2: np.float32(58.813572), 3: np.float32(56.40588), 4: np.float32(76.02489), 5: np.float32(42.017612), 6: np.float32(41.152836), 7: np.float32(128.33417), 8: np.float32(41.621933), 9: 0, 10: np.float32(12.96672)}
